In [84]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-5-5"

In [85]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)

    for block in message.content:
        if block.type == "text":
            return block.text

## 課題: モデルグレーダーを作り、仮のスコア 10 を本物の採点に置き換える

- 対応: L13「モデルベースの採点」。ページのプリフィル方式(```` ```json ```` + `stop_sequences`)は、現行モデルでは使えない可能性が高い(L08)ため、構造化出力(`output_config.format`)で置き換える。
- 今回、新しく覚えること: 採点者を、**別の API 呼び出し**として実装する(生成と採点で、合計2回呼ぶ)。
- 目標: 各テストケースについて、Claude の出力を、別の Claude 呼び出しで採点し、`score` と `reasoning` を結果に入れる。最後に、平均スコアを表示する。
- 採点の観点: ページの3つの基準のうち、「タスクの遵守」(モデルグレーダー向き)を、まず評価する。フォーマットと構文は、次の「コードベースの採点」で扱う。
- ページのコードについて(私の読み): ページの `eval_prompt` は、`{task}` と `{solution}` を差し込む前提で書かれているが、コードを読む限り、差し込む仕組みがない。ここでは、自分で正しく差し込む(振り返りの1で確認する)。
- API を、生成3回 + 採点3回 = 6回呼ぶ。少額の料金がかかる。1分ほどかかることがある。
- ヒントは、下の「ヒント」のセルにある。まず、見ずに、試してみる。

In [86]:
import json
from statistics import mean

# ex07 で作った dataset.json を読み込む(ex08 と同じ)
with open("../ex07_Generating-test-datasets/dataset.json", "r", encoding="utf-8") as f:
    dataset = json.load(f)
print(len(dataset))


# 構造化出力の応答から、本文を取り出す関数(ex06・ex07 と同じ)
def get_text_from_response(response):
    for block in response.content:
        if block.type == "text":
            return block.text
    return None


# ex08 で作った run_prompt(そのまま)
def run_prompt(test_case):
    """プロンプトとテストケースを結合して、Claude の出力を返す"""
    prompt = f"""
次のタスクを解決してください:

{test_case["task"]}
"""
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

3


In [87]:
# TODO 1: 採点結果の JSON スキーマを書く
#   - 全体は object。キーは次の4つ(すべて必須)。
#       "strengths"  : 文字列の配列(良い点)
#       "weaknesses" : 文字列の配列(改善点)
#       "reasoning"  : 文字列(評価の理由)
#       "score"      : 整数(1〜10)
#   - 1〜10 の範囲を、スキーマで指定できるか。L08・L11 のノートの「制約」を思い出す。
grade_schema = {
    "type": "object",
    "properties": {
        "strengths": {"type": "array", "items": {"type": "string"}},
        "weaknesses": {"type": "array", "items": {"type": "string"}},
        "reasoning": {"type": "string"},
        "score": {"type": "integer"},  # TODO: 整数の type 名
    },
    "required": ["strengths", "weaknesses", "reasoning", "score"],  # TODO: 必須にするキー(4つ)
    "additionalProperties": False,
}

In [88]:
# TODO 2: grade_by_model を完成させる
#   - eval_prompt は、f 文字列にして、タスクと解答を差し込む(先頭の f を忘れない)。
#   - 構造化出力で呼び出す(ex06・ex07 と同じ3段の形。schema は TODO 1 の grade_schema)。
#   - 本文を取り出して、json.loads() で辞書にして返す。
def grade_by_model(test_case, output):
    task = test_case["task"]
    eval_prompt = f"""
あなたは熟練のコードレビュアーです。AI が生成した次の解答を評価してください。

タスク: {task}
解答: {output}

次の項目を含めて評価してください。
- strengths: 良い点を1〜3個
- weaknesses: 改善点を1〜3個
- reasoning: 評価の理由を簡潔に
- score: 1〜10 の整数(10が最高、1が最低)
"""  # TODO: 先頭に f を付けて、f 文字列にする

    messages = []
    add_user_message(messages, eval_prompt)

     

    response = client.messages.create(
        model=model,
        max_tokens=4000,
        messages=messages,
        # TODO: output_config=... (grade_schema を渡す)
        output_config={"format": {"type": "json_schema", "schema": grade_schema}}
    )

    print(response.stop_reason, [block.type for block in response.content])
    text = get_text_from_response(response)
    return json.loads(text)  # TODO: text を json.loads() した辞書を返す

In [89]:
# TODO 3: run_test_case を、本物の採点を使う形に書き換える
#   - run_prompt の出力を、grade_by_model に渡して、採点結果を得る。
#   - score と reasoning を、採点結果から取り出す。
#   - 戻り値は、"output"、"test_case"、"score"、"reasoning" の4つを持つ辞書。
def run_test_case(test_case):
    """run_prompt を呼び、モデルグレーダーで採点する"""
    output = run_prompt(test_case)
    model_grade = grade_by_model(test_case, output)  # TODO: grade_by_model を呼ぶ
    score = model_grade["score"]  # TODO: model_grade から "score" を取り出す
    reasoning = model_grade["reasoning"]  # TODO: model_grade から "reasoning" を取り出す
    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

In [90]:
# TODO 4: run_eval に、平均スコアの計算を足す
#   - results の各要素の "score" を集めて、mean() で平均を出す。
#   - 平均を表示して、results を返す。
import statistics

def run_eval(dataset):
    """全ケースを処理し、平均スコアを表示して、結果のリストを返す"""
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = statistics.mean(result["score"] for result in results)  # TODO: results の score の平均
    print(f"Average score: {average_score}")
    return results

In [91]:
# TODO 5: 評価を実行して、結果を表示する(書き換えなくてよい)
#   - output は長いので、score と reasoning を中心に見る。
results = run_eval(dataset)
for r in results:
    print("task:", r["test_case"]["task"][:40], "...")
    print("score:", r["score"])
    print("reasoning:", r["reasoning"])
    print()

end_turn ['thinking', 'text']
end_turn ['text']
end_turn ['thinking', 'text']
Average score: 8.333333333333334
task: S3バケット名が命名規則（小文字・数字・ハイフンのみ、3〜63文字、先頭と末尾は ...
score: 9
reasoning: 要件（小文字・数字・ハイフンのみ、3〜63文字、先頭末尾は英数字）を正規表現で過不足なく表現しており、正しさの観点で問題は見当たらない。fullmatch の選択や境界値テストも適切である。改善点は型安全性、正規表現の可読性、実S3ルールとの差分の扱いといった品質面に限られる。

task: AWS Lambda関数のIAMロールに付与する最小権限のポリシーを表す単一のJ ...
score: 8
reasoning: ポリシー内容は要件を正確に満たしている。Action は s3:PutObject のみ、Resource は my-app-logs バケットの logs/ 配下に限定されており、Version も正しい。一方、出力形式としてはコードブロックと補足説明が付いており、『単一のJSONオブジェクト』という指定から逸脱している。内容の正確性は高いが、形式面の減点を考慮して評価する。

task: CloudWatch Logsのログ行から、AWS EC2インスタンスID（i- ...
score: 8
reasoning: 正規表現は要件を満たし、Python例とLogs Insightsの使用例も動作する妥当なものです。ただし、先頭の `\b` の効果の説明が不正確で、ハイフン直後の `i-` にも一致する点が触れられていません。また、17桁を先に書く理由の説明も本質的ではありません。コードとしての正しさは高く、説明の精度に改善余地があるため、高めの評価としました。



In [92]:
# test_case = dataset[2]

# prompt = f"""
# 次のタスクを解決してください:

# {test_case["task"]}
# """

# messages = []
# add_user_message(messages, prompt)

# response = client.messages.create(
#     model=model,
#     max_tokens=1000,
#     messages=messages,
# )

# print(response.stop_reason)  # TODO: stop_reason を表示する(L03・L07 で見た書き方)
# print(response.usage.output_tokens)  # 出力に使われたトークン数

## ヒント(困ったときだけ開く)

**TODO 1(スキーマ)**
- 整数の type 名は `"integer"`。
- `required` は、4つのキーの名前を、文字列で並べたリスト。
- 1〜10 の範囲を、スキーマで指定する方法: 数値の最小・最大は、構造化出力ではサポートされない(L08 のノート)。範囲は、評価プロンプトの中で伝える。

**TODO 2(`grade_by_model`)**
- f文字列にするには、文字列の開始(引用符の三連続)の直前に `f` を付ける。ex08 の `run_prompt` の書き方を見る。
- 差し込む値の名前は、`{task}` と `{output}`(関数の中の変数名と合わせる)。
- `output_config` の形は、ex06・ex07 と同じ3段: `{"format": {"type": "json_schema", "schema": ...}}`。
- `json.loads(text)` は、文字列を辞書にする。

**TODO 3(`run_test_case`)**
- `grade_by_model(test_case, output)` の戻り値は、辞書。キーで取り出す(`model_grade["score"]`)。

**TODO 4(平均)**
- results の各要素から score だけを集めるには、リスト内包表記: `[result["score"] for result in results]`(初見なら、`for` で `scores` のリストに足していく書き方でもよい)。
- `mean(scores)` で平均が出る(`from statistics import mean` は、すでに書いてある)。

**TODO 5(実行)**
- 書き換えなくてよい。エラーが出たら、どの関数の中で起きたかを、エラーの `--->` の行から探す。

**エラーが出たら**
- 最後の行を読む。400 の場合、`message` に、どの項目が悪いかが書かれていることが多い。
- `TypeError: unsupported operand` や `NoneType` は、どこかの TODO が `None` のまま、という意味のことが多い。

## 振り返り(自分の言葉で書く)

1. 評価プロンプトを、f 文字列にしなかった場合、Claude には、何が送られるか。実際に試して(または考えて)書く。
2. 3件の `score` と `reasoning` を読んで、採点は妥当に見えるか。冗長な出力(見出しやテストコード付き)は、どう評価されたか。
3. `reasoning`・`strengths`・`weaknesses` を一緒に求める理由は何か。ページは、これがないと、モデルが何点前後に落ち着きがちだと言っているか。
4. 同じ出力を、もう一度採点すると、スコアは変わるか。変わるなら、L10 の「採点者の信頼性」の話と、どうつながるか。